# 飞书 Token 手动获取

运行 **Cell 2** 即可拿到 token. 

如果还没有授权码,先按 **Cell 1** 的步骤操作. 

In [28]:
# ============================================================
# 步骤：如何拿到授权码(code)和 state
# ============================================================
# 如果你已经有了 code,直接跳到下一个 cell. 
#
# 1. 生成授权链接(把下面的代码取消注释运行)
#    或者手动拼接：
#    https://accounts.feishu.cn/open-apis/authen/v1/authorize?
#      app_id=cli_a96cff4693b85cc0
#      &redirect_uri=http%3A//localhost%3A8088
#      &scope=offline_access
#      &state=xxxx
#
# 2. 把链接复制到浏览器地址栏,回车打开
#
# 3. 在飞书授权页面,扫码或点击【同意授权】
#
# 4. 浏览器会跳转到一个类似这样的地址：
#    http://localhost:8088/?code=3IKlzc3KCLCy4AE6H9L1edIBx8FADz87&state=wR__4VIGWFWG4drTQ7I4sQ
#
# 5. 从地址栏里复制：
#    - code=  后面那一串(到 & 之前)
#    - state= 后面那一串(到末尾)
#
# 6. 把 code 粘贴到下一个 cell 的 CODE 变量里
# ============================================================

import os, requests, secrets

APP_ID = os.environ.get("FEISHU_APP_ID", "cli_a96cff4693b85cc0")
state = secrets.token_urlsafe(16)

# scope 必须包含 wiki 权限,否则创建知识库报 99991679
scope = (
    "wiki:wiki wiki:wiki:readonly wiki:space:write_only wiki:space:read "
    "wiki:space:retrieve wiki:node:create wiki:node:read wiki:node:update "
    "wiki:node:move wiki:node:copy wiki:node:retrieve wiki:member:create "
    "wiki:member:update wiki:member:retrieve wiki:setting:read wiki:setting:write_only "
    "docs:document.content:read docs:document:copy docs:document:export docs:document:import "
    "docs:document.media:download docs:document.media:upload docs:document.comment:create "
    "docs:document.comment:read docs:document.comment:update docs:document.comment:delete "
    "docs:document.comment:write_only docs:document.subscription docs:document.subscription:read "
    "docs:event:subscribe docs:event.document_deleted:read docs:event.document_edited:read "
    "docs:event.document_opened:read docs:permission.member docs:permission.member:create "
    "docs:permission.member:delete docs:permission.member:update docs:permission.member:readonly "
    "docs:permission.member:retrieve docs:permission.member:transfer docs:permission.member:auth "
    "docs:permission.member:apply docs:permission.setting docs:permission.setting:read "
    "docs:permission.setting:write_only docs:permission.setting:readonly docx:document "
    "docx:document:readonly docx:document.block:convert offline_access "
    "contact:user.id:readonly auth:user.id:read drive:drive drive:file drive:file:readonly "
    "drive:file:upload search:docs:read"
)
 

auth_url = (
    f"https://accounts.feishu.cn/open-apis/authen/v1/authorize?"
    f"app_id={APP_ID}"
    f"&redirect_uri=http%3A//localhost%3A8088"
    f"&scope={requests.utils.quote(scope)}"
    f"&state={state}"
)

print("复制下面链接到浏览器打开：\n")
print(auth_url)
print(f"\nstate: {state}")


复制下面链接到浏览器打开：

https://accounts.feishu.cn/open-apis/authen/v1/authorize?app_id=cli_a96cff4693b85cc0&redirect_uri=http%3A//localhost%3A8088&scope=wiki%3Awiki%20wiki%3Awiki%3Areadonly%20wiki%3Aspace%3Awrite_only%20wiki%3Aspace%3Aread%20wiki%3Aspace%3Aretrieve%20wiki%3Anode%3Acreate%20wiki%3Anode%3Aread%20wiki%3Anode%3Aupdate%20wiki%3Anode%3Amove%20wiki%3Anode%3Acopy%20wiki%3Anode%3Aretrieve%20wiki%3Amember%3Acreate%20wiki%3Amember%3Aupdate%20wiki%3Amember%3Aretrieve%20wiki%3Asetting%3Aread%20wiki%3Asetting%3Awrite_only%20docs%3Adocument.content%3Aread%20docs%3Adocument%3Acopy%20docs%3Adocument%3Aexport%20docs%3Adocument%3Aimport%20docs%3Adocument.media%3Adownload%20docs%3Adocument.media%3Aupload%20docs%3Adocument.comment%3Acreate%20docs%3Adocument.comment%3Aread%20docs%3Adocument.comment%3Aupdate%20docs%3Adocument.comment%3Adelete%20docs%3Adocument.comment%3Awrite_only%20docs%3Adocument.subscription%20docs%3Adocument.subscription%3Aread%20docs%3Aevent%3Asubscribe%20docs%3Aevent.document_

---

## 用 code 换取 token

**操作**：把 `CODE` 改成你从浏览器地址栏复制到的值,然后运行这个 cell. 

In [29]:
import os, json, time, requests
from pathlib import Path

# ========== 从 .env 读取应用凭证 ==========
APP_ID = os.environ.get("FEISHU_APP_ID", "cli_a96cff4693b85cc0")
APP_SECRET = os.environ.get("FEISHU_APP_SECRET", "")
REDIRECT_URI = "http://localhost:8088"
BASE_URL = "https://open.feishu.cn/open-apis"

if not APP_SECRET:
    raise ValueError("从环境变量读取 FEISHU_APP_SECRET 失败,请检查 .env 文件")

# ========== 手动填入授权码 ==========
CODE = "dGKrw54yLEAL4FGfEKde3w7dz9z3Da3E"   # ← 改成你从浏览器地址栏复制的 code

print("🔄 正在用 code 换取 token...\n")

resp = requests.post(
    f"{BASE_URL}/authen/v2/oauth/token",
    headers={"Content-Type": "application/json; charset=utf-8"},
    json={
        "grant_type": "authorization_code",
        "client_id": APP_ID,
        "client_secret": APP_SECRET,
        "code": CODE,
        "redirect_uri": REDIRECT_URI,
    },
    timeout=30,
)

data = resp.json()

if data.get("code", -1) != 0:
    print("❌ 获取失败\n")
    print(f"错误码: {data.get('code')}")
    print(f"错误: {data.get('error')}")
    print(f"描述: {data.get('error_description')}")
else:
    ACCESS_TOKEN = data["access_token"]
    REFRESH_TOKEN = data.get("refresh_token")
    EXPIRES_IN = data["expires_in"]
    SCOPE = data.get("scope", "")

    print("✅ 成功！\n")
    print("access_token:")
    print(f"  {ACCESS_TOKEN}")
    print()
    if REFRESH_TOKEN:
        print("refresh_token:")
        print(f"  {REFRESH_TOKEN}")
    else:
        print("refresh_token: 未获取(需开通 offline_access 权限)")
    print()
    print(f"expires_in : {EXPIRES_IN}s (约 {EXPIRES_IN//3600} 小时)")
    print(f"scope      : {SCOPE}")

    # 保存到本地
    # 统一放到项目目录的 .data/config 下,不落盘到系统目录
    cache_dir = Path("D:/ALL IN AI/MetaBlog/.data/config")
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / "feishu_oauth.json"
    cache = {
        "app_id": APP_ID,
        "access_token": ACCESS_TOKEN,
        "refresh_token": REFRESH_TOKEN,
        "expire_at": time.time() + EXPIRES_IN,
        "scope": SCOPE,
        "saved_at": time.time(),
    }
    cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n💾 已保存到: {cache_path}")


🔄 正在用 code 换取 token...

✅ 成功！

access_token:
  eyJhbGciOiJFUzI1NiIsImZlYXR1cmVfY29kZSI6IkZlYXR1cmVPQXV0aEpXVFNpZ25fQ04iLCJraWQiOiI3NjM0MzM1MzQ1NjMxOTM5NzY0IiwidHlwIjoiSldUIn0.eyJqdGkiOiI3NjM1NTA1MjA2MTQyMTc2MjE0IiwiaWF0IjoxNzc3Nzc5NTkxLCJleHAiOjE3Nzc3ODY3OTEsInZlciI6InYxIiwidHlwIjoiYWNjZXNzX3Rva2VuIiwiY2xpZW50X2lkIjoiY2xpX2E5NmNmZjQ2OTNiODVjYzAiLCJzY29wZSI6ImF1dGg6dXNlci5pZDpyZWFkIGNvbnRhY3Q6dXNlci5pZDpyZWFkb25seSBkb2NzOmRvY3VtZW50LmNvbW1lbnQ6Y3JlYXRlIGRvY3M6ZG9jdW1lbnQuY29tbWVudDpkZWxldGUgZG9jczpkb2N1bWVudC5jb21tZW50OnJlYWQgZG9jczpkb2N1bWVudC5jb21tZW50OnVwZGF0ZSBkb2NzOmRvY3VtZW50LmNvbW1lbnQ6d3JpdGVfb25seSBkb2NzOmRvY3VtZW50LmNvbnRlbnQ6cmVhZCBkb2NzOmRvY3VtZW50Lm1lZGlhOmRvd25sb2FkIGRvY3M6ZG9jdW1lbnQubWVkaWE6dXBsb2FkIGRvY3M6ZG9jdW1lbnQuc3Vic2NyaXB0aW9uIGRvY3M6ZG9jdW1lbnQuc3Vic2NyaXB0aW9uOnJlYWQgZG9jczpkb2N1bWVudDpjb3B5IGRvY3M6ZG9jdW1lbnQ6ZXhwb3J0IGRvY3M6ZG9jdW1lbnQ6aW1wb3J0IGRvY3M6ZXZlbnQuZG9jdW1lbnRfZGVsZXRlZDpyZWFkIGRvY3M6ZXZlbnQuZG9jdW1lbnRfZWRpdGVkOnJlYWQgZG9jczpldmVudC5kb2N1bWVudF9vc

---

## 用 refresh_token 续期(过期后执行,无需重新授权)

In [34]:
import os, json, time, requests
from pathlib import Path

APP_ID = os.environ.get("FEISHU_APP_ID", "cli_a96cff4693b85cc0")
APP_SECRET = os.environ.get("FEISHU_APP_SECRET", "")
BASE_URL = "https://open.feishu.cn/open-apis"
cache_path = Path.home() / f".feishu_oauth_{APP_ID[-8:]}.json"

if not cache_path.exists():
    print("❌ 没有找到缓存文件,请先运行上面的换 token cell")
else:
    cache = json.loads(cache_path.read_text(encoding="utf-8"))
    refresh_token = cache.get("refresh_token")

    if not refresh_token:
        print("❌ 缓存中没有 refresh_token,无法续期")
    else:
        print("🔄 正在刷新 token...\n")
        resp = requests.post(
            f"{BASE_URL}/authen/v2/oauth/token",
            headers={"Content-Type": "application/json; charset=utf-8"},
            json={
                "grant_type": "refresh_token",
                "client_id": APP_ID,
                "client_secret": APP_SECRET,
                "refresh_token": refresh_token,
            },
            timeout=30,
        )
        data = resp.json()

        if data.get("code", -1) != 0:
            print(f"❌ 刷新失败: {data.get('error_description')} (code: {data.get('code')})")
        else:
            cache["access_token"] = data["access_token"]
            cache["refresh_token"] = data.get("refresh_token", refresh_token)
            cache["expire_at"] = time.time() + data["expires_in"]
            cache["saved_at"] = time.time()
            cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")

            print("✅ 刷新成功！\n")
            print("access_token:")
            print(f"  {data['access_token']}")
            print()
            print("refresh_token:")
            print(f"  {data.get('refresh_token', '无')}")
            print(f"\n有效期: {data['expires_in']}s")


🔄 正在刷新 token...

✅ 刷新成功！

access_token:
  eyJhbGciOiJFUzI1NiIsImZlYXR1cmVfY29kZSI6IkZlYXR1cmVPQXV0aEpXVFNpZ25fQ04iLCJraWQiOiI3NjM0MzM1MzQ1NjMxOTM5NzY0IiwidHlwIjoiSldUIn0.eyJqdGkiOiI3NjM1NTA2NDMwNTA5ODQ1NzAxIiwiaWF0IjoxNzc3Nzc5ODc2LCJleHAiOjE3Nzc3ODcwNzYsInZlciI6InYxIiwidHlwIjoiYWNjZXNzX3Rva2VuIiwiY2xpZW50X2lkIjoiY2xpX2E5NmNmZjQ2OTNiODVjYzAiLCJzY29wZSI6ImF1dGg6dXNlci5pZDpyZWFkIG9mZmxpbmVfYWNjZXNzIHdpa2k6c3BhY2U6d3JpdGVfb25seSB3aWtpOndpa2kiLCJhdXRoX2lkIjoiNzYzNTQ3NTM1MjYzMDE1MjE1MCIsImF1dGhfdGltZSI6MTc3Nzc3MjY0MCwiYXV0aF9leHAiOjE4MDkzMDg2NDAsInVuaXQiOiJldV9uYyIsInRlbmFudF91bml0IjoiZXVfbmMiLCJvcGFxdWUiOnRydWUsImVuYyI6IkFpUWtBUUVDQU1JREFBRUJBd0FDQVEwQUF3c0xBQUFBQXdBQUFBZEdaV0YwZFhKbEFBQUFFRzloZFhSb1gyOXdZWEYxWlY5cWQzUUFBQUFJVkdWdVlXNTBTV1FBQUFBQk1BQUFBQVJVYVcxbEFBQUFDakUzTnpjeU5EZ3dNREFQQUFRTUFBQUFBUW9BQVdMRklsbXZnQUFpQ3dBQ0FBQUFERmlRSThlZGQ2cytRcUlmOWdzQUF3QUFBREF0cWhldUp5OVJYS0p3ZHY4WWtwWldXT3JBQ2p0TU00RWIrSzNaK0JFUGtyV3B3d2hra0FQazZHY0lacTN2NC9vQUN3QUZBQUFBQldWMVgyNWpBQjk2eE50MS94NXlhL1

---

## 用 access_token 创建飞书知识库

从本地缓存读取 token,调用飞书 API 创建 Wiki 知识库. 

In [35]:
import os, json, requests
from pathlib import Path
APP_ID = os.environ.get("FEISHU_APP_ID", "cli_a96cff4693b85cc0")
BASE_URL = "https://open.feishu.cn/open-apis"
cache_path = Path("D:/ALL IN AI/MetaBlog/.data/config/feishu_oauth.json")
if not cache_path.exists():
    raise FileNotFoundError(f"没有找到缓存: {cache_path},请先运行上面的换 token cell")
cache = json.loads(cache_path.read_text(encoding="utf-8"))
access_token = cache.get("access_token")
if not access_token:
    raise ValueError("缓存中没有 access_token")
print("🔄 正在创建知识库...\n")
resp = requests.post(
    f"{BASE_URL}/wiki/v2/spaces",
    headers={
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json; charset=utf-8",
    },
    json={
        "name": "测试知识库",
        "description": "通过 API 自动创建的知识库",
    },
    timeout=30,
)
result = resp.json()
print(json.dumps(result, indent=2, ensure_ascii=False))
if result.get("code", -1) == 0:
    space = result.get("data", {}).get("space", {})
    SPACE_ID = space.get("space_id")
    SPACE_URL = f"https://feishu.cn/wiki/space/{SPACE_ID}"
    print(f"\n✅ 创建成功！")
    print(f"   space_id: {SPACE_ID}")
    print(f"   space_url: {SPACE_URL}")
    print(f"   name: {space.get('name')}")
    print(f"\n📌 SPACE_ID 已保存为全局变量,后续 cell 可直接使用")
else:
    print(f"\n❌ 创建失败: {result.get('msg')} (code: {result.get('code')})")


🔄 正在创建知识库...

{
  "code": 0,
  "data": {
    "space": {
      "description": "通过 API 自动创建的知识库",
      "name": "测试知识库",
      "space_id": "7635506440257440696",
      "space_type": "team",
      "visibility": "private"
    }
  },
  "msg": "success"
}

✅ 创建成功！
   space_id: 7635506440257440696
   name: 测试知识库


---

## 在知识库中创建文档节点并追加内容

创建 Wiki 节点(知识库内的文档),然后用 Markdown 内容追加到文档中. 
运行前请确认：
1. 上面的 token cell 已成功执行(access_token 可用)
2. 下面的 SPACE_ID 已替换为实际知识库 ID(可从上面创建知识库的输出复制)

In [32]:
import os, json, requests
from pathlib import Path
APP_ID = os.environ.get("FEISHU_APP_ID", "cli_a96cff4693b85cc0")
BASE_URL = "https://open.feishu.cn/open-apis"
cache_path = Path("D:/ALL IN AI/MetaBlog/.data/config/feishu_oauth.json")
if not cache_path.exists():
    raise FileNotFoundError(f"没有找到缓存: {cache_path}")
cache = json.loads(cache_path.read_text(encoding="utf-8"))
access_token = cache.get("access_token")
if not access_token:
    raise ValueError("缓存中没有 access_token")
# 自动使用前面 cell 创建的 SPACE_ID(如果前面没运行过,请手动设置)
# SPACE_ID = "你的知识库space_id"
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json; charset=utf-8",
}
print("🔄 正在创建 Wiki 文档节点...\n")
# 1. 在知识库中创建文档节点
resp = requests.post(
    f"{BASE_URL}/wiki/v2/spaces/{SPACE_ID}/nodes",
    headers=headers,
    json={
        "title": "测试文档",
        "obj_type": "docx",
        "node_type": "origin",
    },
    timeout=30,
)
result = resp.json()
print(json.dumps(result, indent=2, ensure_ascii=False))
if result.get("code", -1) != 0:
    raise RuntimeError(f"创建节点失败: {result.get('msg')} (code: {result.get('code')})")
node = result["data"]["node"]
NODE_TOKEN = node["node_token"]
OBJ_TOKEN = node["obj_token"]
print(f"\n✅ 节点创建成功！")
print(f"   node_token: {NODE_TOKEN}")
print(f"   obj_token: {OBJ_TOKEN}")
print(f"   知识库链接: https://feishu.cn/wiki/{NODE_TOKEN}")
print(f"   文档链接: https://feishu.cn/docx/{OBJ_TOKEN}")


🔄 正在创建 Wiki 文档节点...

{
  "code": 0,
  "data": {
    "node": {
      "creator": "",
      "has_child": false,
      "node_create_time": "1777779768",
      "node_token": "QxzKwGGbNiukCGkLUUMc8PBnnRh",
      "node_type": "origin",
      "obj_create_time": "1777779768",
      "obj_edit_time": "1777779768",
      "obj_token": "U1MvdoKhwokNdmxUATUcRI8Hn3f",
      "obj_type": "docx",
      "origin_node_token": "QxzKwGGbNiukCGkLUUMc8PBnnRh",
      "origin_space_id": "7635505237368785862",
      "owner": "ou_5fd95247243d26b82a1e1a5fdefe973d",
      "parent_node_token": "GRjawFwT0iIND1ka9DVcyeHunfe",
      "space_id": "7635505237368785862",
      "title": "测试文档"
    }
  },
  "msg": "success"
}

✅ 节点创建成功！
   node_token: QxzKwGGbNiukCGkLUUMc8PBnnRh
   obj_token: U1MvdoKhwokNdmxUATUcRI8Hn3f
   知识库链接: https://feishu.cn/wiki/QxzKwGGbNiukCGkLUUMc8PBnnRh
   文档链接: https://feishu.cn/docx/U1MvdoKhwokNdmxUATUcRI8Hn3f


### 向文档追加 Markdown 内容

将 Markdown 文本转换为飞书 docx block 格式,追加到刚刚创建的文档中. 

In [33]:
from feishu_client import md_to_blocks

markdown_content = """
# 欢迎文档

这是一篇通过 API 自动创建的文档. 

## 特性

- 支持 **粗体**、*斜体*、~~删除线~~
- 支持 `行内代码`
- 支持 [链接](https://feishu.cn)

## 代码示例

```python
print("Hello, Feishu!")
```

## 表格

| 项目 | 状态 |
|------|------|
| API 创建 | ✅ 完成 |
| 内容追加 | ✅ 完成 |

> 这是引用块内容

---

文档创建时间：自动
"""

print("🔄 正在转换 Markdown 并追加到文档...\n")

blocks = md_to_blocks(markdown_content)
print(f"转换完成,共 {len(blocks)} 个 block")

# 追加到文档
resp = requests.post(
    f"{BASE_URL}/docx/v1/documents/{OBJ_TOKEN}/blocks/{OBJ_TOKEN}/children",
    headers=headers,
    json={"children": blocks},
    timeout=30,
)

result = resp.json()
print(json.dumps(result, indent=2, ensure_ascii=False))

if result.get("code", -1) == 0:
    children = result.get("data", {}).get("children", [])
    print(f"\n✅ 追加成功！共写入 {len(children)} 个 block")
else:
    print(f"\n❌ 追加失败: {result.get('msg')} (code: {result.get('code')})")


🔄 正在转换 Markdown 并追加到文档...

转换完成,共 13 个 block
{
  "code": 0,
  "data": {
    "children": [
      {
        "block_id": "doxcnZ8xfpfvB5v17TFThzMti7d",
        "block_type": 3,
        "heading1": {
          "elements": [
            {
              "text_run": {
                "content": "欢迎文档",
                "text_element_style": {
                  "bold": false,
                  "inline_code": false,
                  "italic": false,
                  "strikethrough": false,
                  "underline": false
                }
              }
            }
          ],
          "style": {
            "align": 1,
            "folded": false
          }
        },
        "parent_id": "U1MvdoKhwokNdmxUATUcRI8Hn3f"
      },
      {
        "block_id": "doxcn7b6AckJyuCjUxBoGgPToFc",
        "block_type": 2,
        "parent_id": "U1MvdoKhwokNdmxUATUcRI8Hn3f",
        "text": {
          "elements": [
            {
              "text_run": {
                "content": "这是一篇通过 API 